## 自注意力机制Self-Attention

- RNN 或 LSTM 依靠“逐字递推”的机制，串行计算导致、长距离记忆容易遗忘、无法并行计算
- Transformer让模型能够同时“看到”整句话

自注意力机制的核心在于让输入序列中的每个词都能自主寻找与其他词的关联程度,引入三个矩阵来完成:
* Query ($Q$)：“我要寻找什么？”
* Key ($K$)：“我能提供什么相关信息？”
* Value ($V$)：“我实际包含的具体内容是什么？”

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

公式通俗解读：

   1. $Q \cdot K^T$：让每个词的 Query 去和所有词的 Key 做内积，算出彼此之间的“相关性得分”（注意力分数）。
   2. $\sqrt{d_k}$ 与 $\text{softmax}$：进行缩放和归一化，把得分变成概率分布（权重合为 1）。
   3. 乘以 $V$：用算出来的权重去对 Value 进行加权求和，从而让每个词都融合了整句话中与它相关的上下文知识。


### 用PyTorch实现 单头自注意力Self-Attention 矩阵运算

In [9]:
import torch
import torch.nn.functional as F

# 假设输入一句话包含 3 个词，每个词的向量维度是 4 (seq_len=3, d_model=4)
# 这一步模拟经典 NLP 中的词向量（Word Embeddings）输入
x = torch.randn(3, 4)
print("--- 输入 ---")
print(x)
# 1. 模拟三个线性变换层，初始化 W_q, W_k, W_v 权重矩阵
d_k = 4
W_q = torch.randn(4, d_k)
W_k = torch.randn(4, d_k)
W_v = torch.randn(4, d_k)

# 2. 映射得到 Q, K, V 矩阵
Q = torch.matmul(x, W_q)  # 形状: (3, 4)
K = torch.matmul(x, W_k)  # 形状: (3, 4)
V = torch.matmul(x, W_v)  # 形状: (3, 4)

# 3. 核心步骤 A：计算注意力分数 (Q * K^T) 并进行缩放
# K.t() 是 K 矩阵的转置
scores = torch.matmul(Q, K.t()) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

# 4. 核心步骤 B：通过 Softmax 归一化，得到权重分布
attention_weights = F.softmax(scores, dim=-1)
print("--- 词与词之间的注意力权重矩阵 ---")
print(attention_weights)

# 5. 核心步骤 C：加权求和 Value，输出最终带有上下文信息的特征向量
output = torch.matmul(attention_weights, V)
print("\n--- 最终输出的特征特征向量 ---")
print(output)

--- 输入 ---
tensor([[-1.6348, -1.5878, -1.7563, -0.7798],
        [-1.7453, -1.6638, -1.5389,  0.2995],
        [ 1.0471,  0.1525, -1.7440, -0.9507]])
--- 词与词之间的注意力权重矩阵 ---
tensor([[1.2834e-03, 7.7759e-03, 9.9094e-01],
        [4.7889e-08, 1.9202e-07, 1.0000e+00],
        [1.8017e-01, 8.1983e-01, 3.3216e-09]])

--- 最终输出的特征特征向量 ---
tensor([[ 0.0800,  0.7975, -1.4994, -3.2743],
        [ 0.0571,  0.7910, -1.4868, -3.2941],
        [ 2.6121,  1.5083, -2.8569, -1.1879]])


上面的单头自注意力有一个致命缺陷：模型在同一时间只能聚焦于一种关联关系。

#### 多头注意力机制
为了让大模型具备更强大的特征提取能力，并能记住词语之间的先后顺序。则需要多头注意力机制（Multi-Head Attention）与位置编码（Positional Encoding）。

1. 为什么要“多头”?
    如果只有“一个头”，模型在看“苹果”这个词时，可能只能注意到它是一种“水果”（语义维度）。但如果引入“多个头”，不同的头就可以各自关注不同的维度 ：
    * Header 1：关注主谓宾等语法结构（如：“谁”吃了苹果）。
    * Header 2：关注词与词之间的语义关联（如：苹果属性是红色、脆的）。
    * Header 3：关注长距离指代（如：句子后半部分的“它”指的是苹果）。

2. 矩阵拆分与拼接的艺术
    “多头”并不是真的去训练几组完全独立的网络，而是将高维的空间切分成多个低维子空间。
    假设模型的总隐藏层维度 $d_{model} = 512$，我们设置有 $h = 8$ 个头。

    * 我们不会直接做 512 维的注意力计算。
    * 而是将 $Q, K, V$ 矩阵在特征维度上“切”成 8 份，每一份的维度是 $d_k = 512 / 8 = 64$ 。
    * 这 8 个大小为 64 维的头并行进行自注意力运算，算完之后，把 8 个头的输出在特征维度上重新拼接（Concat）起来，变回 512 维，最后乘上一个输出权重矩阵 $W^o$ 进行特征融合

3. 为什么自注意力机制“天生无序”？
    公式：$\text{Attention}(Q,K,V)$全是矩阵乘法和词与词之间的内积，如果你把输入句子的词语顺序完全打乱（例如把“我吃苹果”改成“苹果吃我”），只要词向量不变，模型算出来的注意力矩阵分布完全是一样的，只是位置换了。

    也就是说，自注意力机制本身是“词袋模型”，完全丧失了语序信息 。为了解决这个问题，必须在输入端强行注入位置信号 。

4. 正余弦位置编码
    Transformer 论文中采用了一种极其优雅的方法：使用不同频率的正弦（Sine）和余弦（Cosine）函数来计算绝对位置编码.
   **为什么要用三角函数？**
   因为三角函数具备相对位置对称性。通过高阶三角函数公式，$PE_{pos+k}$ 可以被 $PE_{pos}$ 线性表出。
   这意味着大模型不仅能知道某个词在第 3 个位置（绝对位置），还能轻易学会第 3 个位置和第 5 个位置之间相隔了 2 个单位（相对位置）。

   最重要的是：位置编码是直接与输入的词向量（Embedding）相加（Add）的，而不是拼接。

#### 用 PyTorch 编写多头注意力层

In [10]:
# 在工业界，大模型的代码里充满了各种张量形状变换。理解 view 和 transpose 是如何控制“多头”在矩阵中流转的。
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除！"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度

        # 定义 Q, K, V 的线性变换矩阵
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # 最后的输出融合矩阵
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        # x 形状: (batch_size, seq_len, d_model)
        batch_size, seq_len, d_model = x.size()

        # 1. 线性映射得到全局的 Q, K, V
        Q = self.W_q(x)  # (batch_size, seq_len, d_model)
        K = self.W_k(x)
        V = self.W_v(x)

        # 2. 核心魔法：将特征维度切分为多头，并转换维度以进行批量矩阵乘法
        # 变换顺序: (B, S, D) -> (B, S, H, D_K) -> (B, H, S, D_K)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # 3. 计算每个头内部的 Scaled Dot-Product Attention
        # 为什么要用 .transpose(1, 2)？如果不做转置直接用 view 会发生什么？（提示：直接 view 会把内存中连续的词语序列关系无脑切断，而转置能保证我们在各个“特征通道”上进行独立的矩阵相乘）。
        # K.transpose(-2, -1) 将最后两维转置，形状变为 (batch_size, num_heads, d_k, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)

        # 归一化得到注意力权重矩阵 (batch_size, num_heads, seq_len, seq_len)
        attn_weights = F.softmax(scores, dim=-1)

        # 与 V 相乘得到混合上下文特征 (batch_size, num_heads, seq_len, d_k)
        context = torch.matmul(attn_weights, V)

        # 4. 逆操作：把 8 个头的结果重新拼回高维空间
        # (B, H, S, D_K) -> (B, S, H, D_K) -> (B, S, D)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        # 5. 通过最终线性层做特征映射
        output = self.W_o(context)

        return output, attn_weights

# --- 测试运行 ---
if __name__ == "__main__":
    # 模拟输入：1个 Batch，句子长 5 个词，每个词 128 维
    sample_input = torch.randn(1, 5, 128)

    # 实例化一个 4 头的注意力机制（每个头 128/4 = 32维）
    mha = MultiHeadAttention(d_model=128, num_heads=4)

    output, weights = mha(sample_input)
    print("输入形状:", sample_input.shape)
    print("输出形状 (应与输入一致):", output.shape)
    print("注意力权重矩阵形状 (Batch, Heads, Seq, Seq):", weights.shape)

输入形状: torch.Size([1, 5, 128])
输出形状 (应与输入一致): torch.Size([1, 5, 128])
注意力权重矩阵形状 (Batch, Heads, Seq, Seq): torch.Size([1, 4, 5, 5])
